In [49]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,       ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
import torch.nn as nn
from torchsummary import summary
from torch.utils.data import Dataset, DataLoader
from helper import fn_plot_torch_hist, fn_plot_confusion_matrix

In [50]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [57]:
RANDOM_STATE = 24 
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


EPOCHS = 25 # number of epochs
BATCH_SIZE = 16
WEIGHT_DECAY=1e-4
ALPHA = 0.001 # learning rate
TRAIN_SIZE = int(BATCH_SIZE*18)
PATIENCE=20
LR_PATIENCE=5
LR_FACTOR=0.1


params = {'legend.fontsize': 'x-large',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'x-large',
          'axes.titlesize':'x-large',
          'xtick.labelsize':'x-large',
          'ytick.labelsize':'x-large'
         }

plt.rcParams.update(params)

CMAP = plt.cm.coolwarm
plt.style.use('seaborn-v0_8-darkgrid')

In [58]:
df= pd.read_csv(r"C:\Users\PGCP-AI\Downloads\ionosphere.data",header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,25,26,27,28,29,30,31,32,33,34
0,1,0,0.99539,-0.05889,0.85243,0.02306,0.83398,-0.37708,1.00000,0.03760,...,-0.51171,0.41078,-0.46168,0.21266,-0.34090,0.42267,-0.54487,0.18641,-0.45300,g
1,1,0,1.00000,-0.18829,0.93035,-0.36156,-0.10868,-0.93597,1.00000,-0.04549,...,-0.26569,-0.20468,-0.18401,-0.19040,-0.11593,-0.16626,-0.06288,-0.13738,-0.02447,b
2,1,0,1.00000,-0.03365,1.00000,0.00485,1.00000,-0.12062,0.88965,0.01198,...,-0.40220,0.58984,-0.22145,0.43100,-0.17365,0.60436,-0.24180,0.56045,-0.38238,g
3,1,0,1.00000,-0.45161,1.00000,1.00000,0.71216,-1.00000,0.00000,0.00000,...,0.90695,0.51613,1.00000,1.00000,-0.20099,0.25682,1.00000,-0.32382,1.00000,b
4,1,0,1.00000,-0.02401,0.94140,0.06531,0.92106,-0.23255,0.77152,-0.16399,...,-0.65158,0.13290,-0.53206,0.02431,-0.62197,-0.05707,-0.59573,-0.04608,-0.65697,g


In [59]:
df.isna().sum().sum()

np.int64(0)

In [60]:
train_df, test_df= train_test_split(df, train_size=TRAIN_SIZE,
                                    stratify=df.iloc[:,-1], random_state=RANDOM_STATE)

In [ ]:
class DataTransformer:
    def __init__(self):
        self.scaler= StandardScaler()
        self.encoder= LabelEncoder()
        self.is_fitted= False

    def fit(self,X,y):
        self.scaler.fit(X)
        self.encoder.fit(y)
        self.is_fitted= True
        return self
    
    def transform(self,X,y):
        if not self.is_fitted:
            raise ValueError('Transformer must be fitted first')
        return self.scaler.transform(X), self.encoder.transform(y)
    
    def fit_transform(self, X,y):
        self.fit(X,y)
        return self.transform(X,y) 

In [62]:
class IonoDS(Dataset):
    def __init__(self,
            dataframe:pd.DataFrame,
            is_train=True,
            label_col=None,
            transformer=None
        ):
        super(IonoDS,self).__init__()
        self.df=dataframe
        self.is_train=is_train
        self.label_col=label_col
        self.transformer=transformer

        y=self.df[label_col].to_numpy()
        X=self.df.drop(label_col,axis=1)

        if self.is_train:
            self.transformer=DataTransformer()
            self.features,self.labels=self.transformer.fit_transform(X,y)
        else:
            if self.transformer is None:
                raise ValueError('Test set requires pre-fitted transformer')
            
            self.features, self.labels=self.transformer.transform(X,y)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):

        feature=self.features[index]
        label=self.labels[index]

        feature = torch(feature, dtype=torch.float32)
        label = self.labels[index]
        
        return feature, label

        

In [64]:
label_col=34

train_ds=IonoDS(train_df, is_train=True, label_col=label_col)

train_loder = DataLoader(
    dataset= train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)
print(train_loder)

In [ ]:
class Model(nn.Module):
    def __init__(self, input_dim):
        super(Model, self).__init__()

        dor1 = 0.05
        dor2 = 0.15
        dor3 = 0.25

        self.layer1 = nn.Linear(input_dim, 26)
        self.bn1 = nn.BatchNorm1d(26)
        self.activ1=nn.ReLU()
        self.do1=nn.Dropout(dor1)

        self.layer2 = nn.Linear(26, 18)
        self.bn2 = nn.BatchNorm1d(18)
        self.activ2=nn.ReLU()
        self.do2=nn.Dropout(dor2)

        self.layer3 = nn.Linear(18, 10)
        self.bn3 = nn.BatchNorm1d(10)
        self.activ3=nn.ReLU()
        self.do3=nn.Dropout(dor3)

        self.layer4 =  nn.Linear(10,2)

    def forward(self, x):
        x=self.do1()